# 02 — TomTom Quantile Calibration

Reproduces the empirical derivation of the p85/p95 travel-time
multipliers from observed TomTom historical-traffic ratios: raw ratio
-> normalization -> pooled empirical quantiles -> multipliers.

Logic lives in `src/quantiles.py`; this notebook orchestrates and
reports both a freshly-computed quantile derivation (when the
appropriate input is available) and the FROZEN experimental multipliers
actually used throughout the study.

**Redistribution note**: permission to redistribute TomTom-derived data
(raw pair-level observations, or even the aggregated ratio-value pool)
has not been confirmed, so **no TomTom-derived file is included in the
public `data_deidentified/` package** -- see `REPRODUCIBILITY_NOTE.md`.
PUBLIC mode therefore reports the frozen, disclosed multiplier values as
a documented design parameter, without independently re-deriving them.
PRIVATE mode performs the full derivation from an authorized copy of the
data placed under `data_private/`.


In [ ]:
import os, sys, json
assert 'REPO_ROOT' in dir(), "Run notebook 00 first."
sys.path.insert(0, REPO_ROOT)
from src.quantiles import (derive_multipliers_from_raw_tomtom, derive_multipliers_from_aggregated_pool,
                            P85_MULTIPLIER, P95_MULTIPLIER, P85_BOOTSTRAP_CI95, P95_BOOTSTRAP_CI95)


## Frozen experimental multipliers (used throughout the study)

In [ ]:
print("FROZEN experimental multipliers (src/quantiles.py):")
print(f"  p50 = 1.0000 (real OSRM base duration, no multiplier)")
print(f"  p85 = {P85_MULTIPLIER}  (bootstrap 95% CI {P85_BOOTSTRAP_CI95})")
print(f"  p95 = {P95_MULTIPLIER}  (bootstrap 95% CI {P95_BOOTSTRAP_CI95})")


## Empirical re-derivation (mode-dependent data availability)

In [ ]:
PRIVATE_DIR = os.path.join(REPO_ROOT, "data_private")
raw_path = os.path.join(PRIVATE_DIR, "tomtom_raw_validation.csv")
aggregated_path = os.path.join(PRIVATE_DIR, "tomtom_ratio_pool_aggregated_PRIVATE.csv")

computed = None
if DATA_MODE == "private" and os.path.exists(raw_path):
    print(f"PRIVATE mode: recomputing from raw pair-level data ({raw_path}), "
          f"with pair-clustered bootstrap.")
    computed = derive_multipliers_from_raw_tomtom(raw_path)
    print(json.dumps(computed, indent=2))
elif os.path.exists(aggregated_path):
    print(f"Recomputing from the aggregated ratio-value pool ({aggregated_path}), "
          f"with an observation-level (not pair-clustered) bootstrap.")
    computed = derive_multipliers_from_aggregated_pool(aggregated_path)
    print(json.dumps(computed, indent=2))
else:
    print("No TomTom-derived input available in this environment.")
    print("PUBLIC mode: this is EXPECTED -- redistribution permission for TomTom-derived")
    print("data has not been confirmed, so no such file ships in data_deidentified/.")
    print("The frozen multipliers above are reported as a disclosed design parameter,")
    print("not independently re-verified numerically in this mode.")
    print("PRIVATE mode: place an authorized copy under data_private/ to enable this step.")


## Programmatic verification against frozen values (only when computed)

Hard assertion, not just a side-by-side print: if a re-derivation was
possible above, its p85/p95 MUST match the frozen constants within a
small tolerance.


In [ ]:
TOLERANCE = 0.01
if computed is not None:
    p85_diff = abs(computed['p85'] - P85_MULTIPLIER)
    p95_diff = abs(computed['p95'] - P95_MULTIPLIER)
    print(f"p85: computed={computed['p85']:.4f} vs frozen={P85_MULTIPLIER} (diff={p85_diff:.4f})")
    print(f"p95: computed={computed['p95']:.4f} vs frozen={P95_MULTIPLIER} (diff={p95_diff:.4f})")
    assert p85_diff < TOLERANCE, f"p85 mismatch exceeds tolerance {TOLERANCE}"
    assert p95_diff < TOLERANCE, f"p95 mismatch exceeds tolerance {TOLERANCE}"
    print(f"\nVerification PASSED (within tolerance {TOLERANCE}).")
else:
    print("Skipped: no TomTom-derived input available in this environment/mode "
          "(see explanation above -- this is an expected, documented state, not a failure).")


## Expected outputs / integrity checks

In [ ]:
checks = {}
if computed is not None:
    checks["p85_within_tolerance"] = abs(computed['p85'] - P85_MULTIPLIER) < TOLERANCE
    checks["p95_within_tolerance"] = abs(computed['p95'] - P95_MULTIPLIER) < TOLERANCE
else:
    checks["input_unavailable_documented_not_silently_skipped"] = True

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_02_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 02 STATUS: {NOTEBOOK_02_STATUS}")
assert NOTEBOOK_02_STATUS == "PASS"


## Save output

In [ ]:
results_dir = os.path.join(REPO_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)
out = {
    "frozen_multipliers": {"p50": 1.0, "p85": P85_MULTIPLIER, "p95": P95_MULTIPLIER},
    "frozen_bootstrap_ci95": {"p85": P85_BOOTSTRAP_CI95, "p95": P95_BOOTSTRAP_CI95},
    "computed_full_precision": computed,
    "tomtom_input_available": computed is not None,
}
with open(os.path.join(results_dir, "tomtom_quantile_calibration.json"), "w") as f:
    json.dump(out, f, indent=2)
print("Saved results/tomtom_quantile_calibration.json")
